# 1. Decay

Part of the **[Fig. 4 chapter](fig4.md)** — see it for the panel-by-panel map, run order, and data inputs. The first code cell sets `ENTEX_ROOT` and activates the no-overwrite guard (see the [Reproduction guide](../reproduce.md)).


In [ ]:
# === Reproduction setup — edit ENTEX_ROOT / REF_ROOT for your machine ===
# All absolute paths below resolve from these two roots; the working directory
# is the original analysis/ folder, and repro_guard prevents any existing file
# from being overwritten when you re-run this notebook. See the book's
# 'Reproduction guide' chapter for details.
import os, sys
ENTEX_ROOT = os.environ.get('ENTEX_ROOT', '/large_storage/zhoulab/zhoujt/project/ENTEx')
REF_ROOT   = os.environ.get('REF_ROOT',   '/large_storage/zhoulab/ref')
BOOK_ROOT  = os.environ.get('BOOK_ROOT',  f'{ENTEX_ROOT}/analysis/HumanCellEpigenomeAtlas')
sys.path.insert(0, BOOK_ROOT)
os.chdir(f'{ENTEX_ROOT}/analysis')   # original working directory
import repro_guard                    # no-overwrite guard (default: skip existing)


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
import anndata
import scanpy as sc
from glob import glob
import time
from scipy.stats import spearmanr, pearsonr
from itertools import cycle, islice

mpl.style.use('default')
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = 'Helvetica'


In [ ]:
indir = f'{ENTEX_ROOT}/'
outdir = f'{ENTEX_ROOT}/analysis/'


In [ ]:
# group_meta = pd.read_csv(f'{indir}clustering/merged/group_meta.tsv', sep='\t', header=0, index_col=0)
# # group_meta = group_meta[['L2_any', 'L1', 'count']]
# group_meta['L1_annot'] = group_meta['L1_annot'].str.replace(' ','-').str.replace('/','_')
# annot2L1 = group_meta[['L1','L1_annot']].set_index('L1_annot')['L1'].to_dict()
# L1annot = group_meta[['L1','L1_annot']].set_index('L1')['L1_annot'].to_dict()


In [ ]:
meta = anndata.read_h5ad(f'{indir}clustering/merged/5kCG100k3C_summary.h5ad').obs
# meta['L1'] = meta['L1'].astype(str)
# meta.loc[meta['L1']=='c7', 'L1'] = 'c35'
# meta.loc[(meta['L1']=='c35') & meta['L2_any'].isin(['c0','c10','c11']), 'L1'] = 'c36'


In [ ]:
decay = pd.concat([pd.read_hdf(xx, key='data') for xx in glob(f'{outdir}decay/cell_*_decay.hdf5')], axis=0)
decay = decay.loc[meta.index]
decay = decay / decay.sum(axis=1).values[:,None]


In [ ]:
L1_meta = pd.read_csv(f'{indir}L1color.tsv', sep='\t', header=0, index_col=0)
L1_meta = L1_meta.drop(['c35','c36'], axis=0)
L1_annot = L1_meta['L1_abbr'].to_dict()
L1_color = L1_meta['color'].to_dict()


In [ ]:
(np.log2([200000,2000000,10000000,100000000])-np.log2(2500))/0.125

In [ ]:
meta['short'] = decay.loc[:, (decay.columns > 50) & (decay.columns < 78)].mean(axis=1)
meta['long'] = decay.loc[:, (decay.columns > 95) & (decay.columns < 123)].mean(axis=1)
meta['ratio'] = np.log2(meta['short'] / meta['long'])


In [ ]:
row_colors = meta['L1'].map(L1colors)
selc = meta.sort_values('ratio').index
cg = sns.clustermap(decay.loc[selc], metric='cosine', z_score=0, vmin=-2, vmax=4, 
                    col_cluster=False, row_cluster=False, cmap='Blues', 
                    row_colors=row_colors, figsize=(10,12))

ax = cg.ax_heatmap
xticks = [5000, 10000, 20000, 50000, 100000, 200000, 500000, 1000000, 2000000, 5000000, 10000000, 20000000, 50000000, 100000000]
xticklabels = ['5k', '10k', '20k', '50k', '100k', '200k', '500k', '1M', '2M', '5M', '10M', '20M', '50M', '100M']
ax.set_xticks([np.log2(xx/2500)/0.125 for xx in xticks])
ax.set_xticklabels(xticklabels, rotation=90, ha='center')
for xx in L1colors.index:
    cg.ax_col_dendrogram.bar(0, 0, color=L1colors.loc[xx], label=L1annot[xx], linewidth=0)
    
cg.ax_col_dendrogram.legend(loc="center", ncol=5, framealpha=0)
for xx in [200000,2000000,10000000,100000000]:
    ax.vlines(x=np.log2(xx/2500)/0.125, colors='k', ymin=-0.5, ymax=decay.shape[0]-0.5, linewidths=0.5)


In [ ]:
leg = meta.groupby('L1')['ratio'].median().sort_values().index[::-1]
selc = []
for xx in leg:
    tmp = meta.loc[meta['L1']==xx]
    selc.append(tmp.sort_values('ratio').index[::-1])
    
selc = np.concatenate(selc)
meta = meta.loc[selc]
decay = decay.loc[selc]


In [ ]:
offset = [0]
count = meta['L1'].value_counts()
for xx in leg:
    offset.append(offset[-1]+count.loc[xx])

offset = np.array(offset)


In [ ]:
import matplotlib.patches as patches
from scipy.stats import zscore

fig, axes = plt.subplots(5, 1, gridspec_kw={'height_ratios': [20,1,1,1,1]}, figsize=(10,5), sharex='all', dpi=300)
ax = axes[0]
ax.imshow(zscore(decay, axis=1).T, cmap='Oranges', vmin=0, vmax=4, 
          aspect='auto', interpolation='none', rasterized=True)
ax.set_xticks([])
xticks = [5000, 50000, 500000, 5000000, 50000000]
xticklabels = ['5k', '50k', '500k', '5M', '50M']
ax.set_yticks([np.log2(xx/2500)/0.125 for xx in xticks])
ax.set_yticklabels(xticklabels, fontsize=12)
for xx in offset:
    ax.vlines(x=xx-0.5, colors='k', ymin=-0.5, ymax=decay.shape[1]-0.5, linewidths=0.5)

ax = axes[1]
ax.axis('off')
# ax.set_ylabel('MajorType', rotation=0)
for i,xx in enumerate(leg):
    rect = patches.Rectangle((offset[i], 0), offset[i+1]-offset[i], 1, linewidth=0, edgecolor='none', facecolor=L1_color[xx])
    ax.add_patch(rect)
    # ax.text(np.mean(offset[i:(i+2)]), -0.2, label, rotation=90, fontsize=10, horizontalalignment='left', verticalalignment='top')

ax = axes[2]
ax.imshow(meta['mCGFrac'].values[None, :], aspect='auto', vmin=0.65, vmax=0.85, 
          interpolation='none', cmap='Blues', rasterized=True)
ax.axis('off')

ax = axes[3]
ax.imshow(meta['mCHFrac'].values[None, :], aspect='auto', vmin=0.005, vmax=0.025, 
          interpolation='none', cmap='Purples', rasterized=True)
ax.axis('off')

ax = axes[4]
ax.imshow(np.log2(meta['Cis/Trans'].values[None, :]), aspect='auto', vmin=0, vmax=1.5, 
          interpolation='none', cmap='viridis', rasterized=True)
ax.axis('off')

for ax,xx in zip(axes, ['Contact Distance', 'MajorType', 'Global mCG', 'Global mCH', 'Cis/Trans']):
    ax.set_title(xx, fontsize=15)

plt.tight_layout()
plt.savefig(f'decay/cell_{meta.shape[0]}_majortype_decay.pdf', transparent=True, dpi=300)



In [ ]:
import matplotlib.patches as patches
from scipy.stats import zscore

tmp = zscore(decay, axis=1).T

fig, axes = plt.subplots(1, 35, gridspec_kw={'wspace':0}, figsize=(8,2), dpi=300)
for i,ax in enumerate(axes):
    ax.imshow(tmp.iloc[:, offset[i]:offset[i+1]], cmap='Oranges', vmin=0, vmax=3, 
              aspect='auto', interpolation='none', rasterized=True)
    ax.set_xticks([])
    ax.set_yticks([])

ax = axes[0]
yticks = [5000, 50000, 500000, 5000000, 50000000]
yticklabels = ['5k', '50k', '500k', '5M', '50M']
ax.set_yticks([np.log2(xx/2500)/0.125 for xx in yticks])
ax.set_yticklabels(yticklabels, fontsize=12)

fig.tight_layout()
fig.savefig(f'decay/cell_{meta.shape[0]}_majortype_decay_equal.pdf', transparent=True, dpi=300)



In [ ]:
from scipy.stats import pearsonr
pearsonr(meta['Cis/Trans'], meta['ratio'])

In [ ]:
fig, ax = plt.subplots(figsize=(8,3), dpi=300)
sns.boxplot(data=meta, x='L1', y='ratio', order=leg, ax=ax, palette=L1_color, showfliers=False)
ax.set_xticklabels([L1_annot[xx] for xx in leg], rotation=-90)
plt.tight_layout()
plt.savefig(f'decay/cell_{meta.shape[0]}_majortype_decay_boxplot.pdf', transparent=True)


In [ ]:
L1_annot['c21'] = 'Glia Schw'

In [ ]:
germ_layer = {
    # Endothelium / vasculature
    "Endo Lymphatic": "mesoderm",
    "Endo Vascular": "mesoderm",

    # Epithelia
    "Epi Acinar": "endoderm",              # e.g. pancreatic / digestive acini
    "Epi Adrenal": "mesoderm",            # adrenal cortex (epithelioid steroidogenic cells)
    "Epi Alveolar": "endoderm",
    "Epi Breast Basal": "ectoderm",
    "Epi Ductal": "ectoderm",             # mammary duct epithelium
    "Epi Endocrine": "endoderm",          # gut/pancreatic endocrine epithelia
    "Epi Enteric": "endoderm",
    "Epi Gastric": "endoderm",
    "Epi Keratinous/Luminal": "ectoderm", # epidermis / keratinocytes
    "Epi Trophoblast": "ectoderm",        # strictly trophectoderm/extraembryonic, but ectoderm-derived

    # Fibroblasts / myofibroblasts
    "Fibro Adrenal": "mesoderm",
    "Fibro Breast": "mesoderm",
    "Fibro Endoneurial": "mesoderm",
    "Fibro Epineurial": "mesoderm",
    "Fibro Gastrointestinal": "mesoderm",
    "Fibro Heart": "mesoderm",
    "Fibro Muscular": "mesoderm",
    "Myofibroblast": "mesoderm",
    "Fibro Skin": "mesoderm",

    # Glia / neurons
    "Glia Astrocyte": "ectoderm",
    "Glia Oligodendrocyte": "ectoderm",
    "Neu Excitatory": "ectoderm",
    "Neu Inhibitory": "ectoderm",
    "Glia Schwann": "ectoderm",            # neural crest–derived

    # Hematopoietic / immune
    "Hema B": "mesoderm",
    "Hema Mast": "mesoderm",
    "Hema Myeloid": "mesoderm",
    "Hema NK": "mesoderm",
    "Hema Tmem": "mesoderm",
    "Hema Tnaive": "mesoderm",

    # Muscle / mural cells
    "Mus Cardiac": "mesoderm",
    "Mus Skeletal": "mesoderm",
    "Perivascular": "mesoderm",           # pericytes / mural cells
}

In [ ]:
ratio_df = pd.DataFrame(meta.groupby('L1')['ratio'].median())
ratio_df['L1_annot'] = ratio_df.index.map(L1_annot)
ratio_df['L0'] = ratio_df['L1_annot'].str.split(' ').str[0]
ratio_df['germ_layer'] = ratio_df.index.map(L1_meta['L1_annot'].to_dict()).map(germ_layer)
ratio_df['L0'].value_counts()

In [ ]:
fig, ax = plt.subplots(figsize=(3,3), dpi=300)
sns.stripplot(data=ratio_df, x='L0', y='ratio', hue='L1', 
              order=ratio_df.groupby('L0')['ratio'].mean().sort_values().index,
              palette=L1_color, s=4, edgecolor='none', legend=False, ax=ax)
ax.set_xticklabels(ax.get_xticklabels(), rotation=90)
ax.set_xlabel('')
ax.set_ylabel('log2 Short/Long')
fig.tight_layout()
fig.savefig(f'decay/L0_decay_stripplot.pdf', transparent=True)

In [ ]:
import itertools
from scipy.stats import ranksums
for ct1,ct2 in itertools.combinations(ratio_df['L0'].unique(), 2):
    data1 = ratio_df.loc[ratio_df['L0']==ct1, 'ratio']
    data2 = ratio_df.loc[ratio_df['L0']==ct2, 'ratio']
    if (len(data1) < 2) or (len(data2) < 2):
        continue
    stat, pval = ranksums(data1, data2)
    if pval < 0.05:
        print(f'{ct1} vs {ct2}: s={stat:.3f}, p={pval:.3e}')

In [ ]:
import itertools
from scipy.stats import ranksums
from statsmodels.sandbox.stats.multicomp import multipletests as FDR

pv = []
mt = []
for ct in ratio_df['L0'].unique():
    data1 = ratio_df.loc[ratio_df['L0']==ct, 'ratio']
    data2 = ratio_df.loc[ratio_df['L0']!=ct, 'ratio']
    if (len(data1) < 2) or (len(data2) < 2):
        continue
    stat, pval = ranksums(data1, data2)
    if pval < 0.05:
        print(f'{ct}: s={stat:.3f}, p={pval:.3e}')
    pv.append(pval)
    mt.append(ct)

fdr = FDR(pv, 0.01, "fdr_bh")[1]
pd.Series(fdr, index=mt).sort_values()


In [ ]:
adata = anndata.read_h5ad(f'{indir}clustering/merged/5kCG100k3C_summary.h5ad')
adata.obs['Short/Long'] = meta['ratio'].copy()
adata.write_h5ad(f'{indir}clustering/merged/5kCG100k3C_summary.h5ad')


In [ ]:
rnameta = rnameta.loc[data.index[data['Study']=='10x']]
rnameta['pred_from_mC'] = data.loc[rnameta.index, 'pred_from_mC'].values


In [ ]:
rnameta.groupby('pred_from_mC')['BarcodeTotalUMIs'].mean().sort_values()


In [ ]:
meta.groupby('MajorType')['ratio'].mean().sort_values()

In [ ]:
from scipy.stats import pearsonr
pearsonr(meta.groupby('MajorType')['ratio'].mean().loc[leg], rnameta.groupby('pred_from_mC')['BarcodeTotalUMIs'].mean().loc[leg])


In [ ]:
fig, ax = plt.subplots(figsize=(4,4))
ax.scatter(rnameta.groupby('pred_from_mC')['BarcodeTotalUMIs'].mean().loc[leg], meta.groupby('MajorType')['ratio'].mean().loc[leg], c=pd.Series(leg).map(colordict))
for xx in leg:
    ax.text(rnameta.groupby('pred_from_mC')['BarcodeTotalUMIs'].mean().loc[xx], meta.groupby('MajorType')['ratio'].mean().loc[xx], xx)

ax.set_xlabel('Total 10x UMI')
ax.set_ylabel('Short/Long Ratio')
plt.savefig(f'{indir}/plot/celltype_ratio_rnacount_corr.pdf', transparent=True)


In [ ]:
motifad.obs['dmr'] = ['_'.join(xx.split('_')[:-1]) for xx in motifad.obs.index]


In [ ]:
ctmotif = pd.DataFrame(index=motifad.var.index, columns=leg)
count = {}
for xx in leg:
    ctmotif[xx] = motifad.X[motifad.obs[xx].values].sum(axis=0).A1
    count[xx] = motifad.obs[xx].sum()
    print(xx)
    

In [ ]:
ctmotif

In [ ]:
ctmotif = ctmotif / pd.Series(count)

In [ ]:
ctmotif.loc['MA0139.1']

In [ ]:
meta.groupby('MajorType')['ratio'].mean().loc[leg]

In [ ]:
from scipy.stats import spearmanr
pearsonr(meta.groupby('MajorType')['ratio'].mean().loc[leg], ctmotif.loc['MA0139.1'])


In [ ]:
ratio = meta.groupby(hue)['ratio'].mean().loc[leg]


In [ ]:
leg0 = ['L23_IT', 'L4_IT', 'L5_IT', 'L6_IT', 'L6_IT_Car3', 'L56_NP', 'L6_CT', 'L6b', 'L5_ET', 'Amy', 
       'Lamp5', 'Lamp5_LHX6', 'Sncg', 'Vip', 'Pvalb', 'Pvalb_ChC', 'Sst', 'CHD7', 
       'MSN_D1', 'MSN_D2', 'Foxp2', 'SubCtx', 
       'ASC', 'ODC', 'OPC', 'MGC', 'PC', 'EC', 'VLMC'
      ]
legname = ['L2/3-IT', 'L4-IT', 'L5-IT', 'L6-IT', 'L6-IT-Car3', 'L5/6-NP', 'L6-CT', 'L6b', 'L5-ET', 'Amy-Exc', 
       'Lamp5', 'Lamp5-Lhx6', 'Sncg', 'Vip', 'Pvalb', 'Pvalb-ChC', 'Sst', 'Chd7', 
       'MSN-D1', 'MSN-D2', 'Foxp2', 'SubCtx-Cplx', 
       'ASC', 'ODC', 'OPC', 'MGC', 'PC', 'EC', 'VLMC'
      ]



In [ ]:
sad = np.load(f'{indir}compartment_majortype/saddle_impute_mergerawpca.npy')
ws = 10
compstr = [[sad[i][:ws, :ws].mean(), sad[i][-ws:, -ws:].mean(), 
            np.mean([sad[i][:ws, -ws:].mean(), sad[i][-ws:, :ws].mean()])]
            for i in range(len(leg)+1)]
compstr = pd.DataFrame(compstr, index=leg0+['merged'], columns=['BB', 'AA', 'Inter'])
compstr['Strength'] = compstr[['BB','AA']].mean(axis=1) / compstr['Inter']


In [ ]:
# impute

fig, axes = plt.subplots(1, 4, figsize=(14,3), sharey='all')
for i,x in enumerate(compstr.columns):
    ax = axes[i]
    ax.scatter(compstr.loc[leg, x], ratio, c=pd.Series(leg).map(colordict))
#     for k,xx in enumerate(leg0):
#         ax.text(compstr.loc[xx, x], ratio.loc[xx], legname[k])
    ax.set_title(f'r={np.around(pearsonr(compstr.loc[leg, x], ratio)[0], decimals=3)}', fontsize=10)
    ax.set_xlabel(x, fontsize=10)
    
# plt.tight_layout()
plt.savefig(f'{indir}/plot/celltype_ratio_compstr_impute_corr.pdf', transparent=True)


In [ ]:
for i,x in enumerate(compstr.columns):
    print(pearsonr(compstr.loc[leg, x], ratio))


In [ ]:
sad = np.load(f'{indir}compartment_majortype/saddle_raw_mergerawpca.npy')
ws = 10
compstr = [[sad[i][:ws, :ws].mean(), sad[i][-ws:, -ws:].mean(), 
            np.mean([sad[i][:ws, -ws:].mean(), sad[i][-ws:, :ws].mean()])]
            for i in range(len(leg)+1)]
compstr = pd.DataFrame(compstr, index=leg0+['merged'], columns=['BB', 'AA', 'Inter'])
compstr['Strength'] = compstr[['BB','AA']].mean(axis=1) / compstr['Inter']


In [ ]:
# raw

fig, axes = plt.subplots(1, 4, figsize=(14,3), sharey='all')
for i,x in enumerate(compstr.columns):
    ax = axes[i]
    ax.scatter(compstr.loc[leg, x], ratio, c=pd.Series(leg).map(colordict))
#     for k,xx in enumerate(leg0):
#         ax.text(compstr.loc[xx, x], ratio.loc[xx], legname[k])
    ax.set_title(f'r={np.around(pearsonr(compstr.loc[leg, x], ratio)[0], decimals=3)}', fontsize=10)
    ax.set_xlabel(x, fontsize=10)
    
# plt.tight_layout()
plt.savefig(f'{indir}/plot/celltype_ratio_compstr_raw_corr.pdf', transparent=True)



In [ ]:
for i,x in enumerate(compstr.columns):
    print(pearsonr(compstr.loc[leg, x], ratio))


In [ ]:
fig, ax = plt.subplots(figsize=(3,2), dpi=300)
ax.axis('off')
coordx = np.repeat(np.arange(3),10)[:-1]
coordy = np.tile(np.arange(10), 3)[:-1]
ax.scatter(coordx-0.1, coordy, c=[colordict[xx] for xx in leg0])
for i,xx in enumerate(legname):
    ax.text(coordx[i], coordy[i], xx, ha='left', va='center')

ax.invert_yaxis()
plt.savefig(f'{indir}/plot/celltype_legend.pdf', transparent=True)


## Liu 2021

In [ ]:
from wmb import cemba
meta = cemba.get_m3c_mapping_metric()
meta

In [ ]:
contact_table = pd.read_csv(cemba.CEMBA_SNM3C_CONTACT_PATH, index_col=0)
contact_table = contact_table[contact_table['0'].str.split('/').str[6].isin([f'HIP{i+1}' for i in range(4)])]


In [ ]:
import xarray as xr
annot = xr.open_zarr(f'{cemba.CEMBA_SNM3C_CELL_TYPE_ANNOTATION_PATH}/CEMBA.snm3C.Annotations.zarr/')
annot

In [ ]:
selc = np.sort(annot.get_index('cell').intersection(meta.index & contact_table.index))
print(len(selc))


In [ ]:
meta = meta.loc[selc]
meta['CellType'] = annot['CellType'].to_pandas().loc[selc]
contact_table = contact_table.loc[selc]


In [ ]:
contact_table.to_csv(f'{indir}contact_table.tsv', sep='\t', header=None)


In [ ]:
# decay = pd.concat([pd.read_hdf(xx, key='data') for xx in decay_list], axis=0)
decay = pd.concat([pd.read_hdf(xx, key='data') for xx in glob(f'{indir}cell_*_decay.hdf5')], axis=0)
decay


In [ ]:
count = meta['CellType'].value_counts()

selc = decay.index & meta.index[meta['CellType'].isin(count.index[count>20])]
decay = decay.loc[selc]
meta = meta.loc[selc]
print(meta.shape)

In [ ]:
decay = decay / decay.sum(axis=1)[:,None]


In [ ]:
hue = 'CellType'
# colordict = hba_data.internal.celltype.CellType.majortype_palette()
count = count[count>20]
colordict = {xx:yy for xx,yy in zip(count.index, sns.color_palette('tab20', count.shape[0]))}
rowcolor = [colordict[xx] for xx in meta[hue]]
# rowcolor = np.array([rowcolor, meta['mCHFrac'].values]).T


In [ ]:
np.log2(np.array([100000, 1500000, 20000000, 50000000])/2500)/0.125

In [ ]:
meta['short'] = decay.loc[:, (decay.columns > 42) & (decay.columns < 74)].mean(axis=1)
meta['long'] = decay.loc[:, (decay.columns > 103) & (decay.columns < 114)].mean(axis=1)
meta['ratio'] = np.log2(meta['short'] / meta['long'])


In [ ]:
leg0 = count.index


In [ ]:
len(leg0)

In [ ]:
selc = meta.sort_values('ratio').index
g = sns.clustermap(decay.loc[selc], z_score = 0, metric='cosine', cmap="Blues", col_cluster=False, figsize=(10,10),
                   row_cluster=False, vmin=-2, row_colors=meta.loc[selc, hue].map(colordict))
ax = g.ax_heatmap
ax.set_yticks([])
ax.set_xticks([np.log2(xx/2500)/0.125 for xx in [5000, 10000, 20000, 50000, 100000, 200000, 500000, 1000000, 2000000, 5000000, 10000000, 20000000, 50000000]])
ax.set_xticklabels(['5k', '10k', '20k', '50k', '100k', '200k', '500k', '1M', '2M', '5M', '10M', '20M', '50M'], rotation=90, ha='center')

for label in leg0:
    g.ax_col_dendrogram.bar(0, 0, color=colordict[label],
                            label=label, linewidth=0)
g.ax_col_dendrogram.legend(loc="center", ncol=4, framealpha=0, bbox_to_anchor=(0.5, 0.5))
# g.savefig(f'{indir}/plot/cell_{meta.shape[0]}_sort_decay.png', transparent=True)



In [ ]:
meta = meta.sample(frac=1)


In [ ]:
leg = meta.groupby(hue)['ratio'].median().sort_values().index[::-1]


In [ ]:
# leg = ['Neuron', 'Neonatal Neuron 1', 'Neonatal Neuron 2', 
#        'Cortical L2–5 Pyramidal Cell', 'Cortical L6 Pyramidal Cell', 'Interneuron', 'Medium Spiny Neuron',
#        'Hippocampal Pyramidal Cell', 'Hippocampal Granule Cell',
#        'Neonatal Astrocyte', 'Adult Astrocyte', 
#        'Oligodendrocyte Progenitor', 'Mature Oligodendrocyte', 'Microglia Etc.',
#        'Unknown'
#       ]

meta = pd.concat([meta[meta[hue]==xx] for xx in leg], axis=0)
meta


In [ ]:
decay = decay.loc[meta.index]


In [ ]:
offset = [0]
count = meta[hue].value_counts().loc[leg].values
for i in range(len(leg)):
    offset.append(offset[-1]+count[i])

offset = np.array(offset)


In [ ]:
meta['Cis/Trans'] = meta['CisLongContact'] / meta['TransContact']

In [ ]:
import matplotlib.patches as patches
from scipy.stats import zscore

fig, axes = plt.subplots(3, 1, gridspec_kw={'height_ratios': [20,1,1]}, figsize=(5,3), sharex='all')
ax = axes[0]
ax.imshow(zscore(decay, axis=1).T, cmap='Blues', aspect='auto', interpolation='none', vmin=-2, vmax=4)
ax.set_xticks([])
ax.set_yticks([np.log2(xx/2500)/0.125 for xx in [5000, 20000, 100000, 500000, 2000000, 10000000, 50000000]])
ax.set_yticklabels(['5k', '20k', '100k', '500k', '2M', '10M', '50M'])

# ax = axes[1]
# ax.imshow(meta['Cis/Trans'].values[:,None], aspect='auto', interpolation='none', cmap='bwr')
# ax.axis('off')

# ax = axes[2]
# ax.imshow(meta['mCHFrac'].values[:,None], aspect='auto', interpolation='none', vmax=0.03)
# ax.axis('off')

ax = axes[1]
ax.axis('off')
# ax.set_ylabel('MajorType', rotation=0)
for i,label in enumerate(leg):
    rect = patches.Rectangle((offset[i], 0), offset[i+1]-offset[i], 1, linewidth=0, edgecolor='none', facecolor=colordict[label])
    ax.add_patch(rect)
    # ax.text(np.mean(offset[i:(i+2)]), -0.2, label, rotation=90, fontsize=10, horizontalalignment='left', verticalalignment='top')

ax = axes[2]
ax.imshow(np.log2(meta['Cis/Trans'][None, :]), vmin=0, vmax=2, cmap='viridis', aspect='auto', interpolation='none')
ax.axis('off')

for ax,xx in zip(axes, ['Contact Distance', 'MajorType', 'Cis/Trans']):
    ax.set_title(xx, fontsize=10)

plt.tight_layout()
plt.savefig(f'{indir}../plot/cell_{meta.shape[0]}_majortype_decay.pdf', transparent=True, dpi=300)



In [ ]:
f'{indir}../plot/cell_{meta.shape[0]}_majortype_decay.pdf'

In [ ]:
from scipy.stats import pearsonr
pearsonr(meta['Cis/Trans'], meta['ratio'])

In [ ]:
fig, ax = plt.subplots(figsize=(6,5))
sns.boxplot(data=meta, x=hue, y='ratio', order=leg, ax=ax, palette=colordict, showfliers=False)
ax.set_xticklabels(ax.get_xticklabels(), rotation=90)
plt.tight_layout()
plt.savefig(f'{indir}../plot/cell_{meta.shape[0]}_majortype_decay_boxplot.pdf', transparent=True, dpi=300)
